In [2]:
data_set = "spam_or_not_spam.csv"
text_col = "email"
label_col = "label"

model_name = "distilbert-base-uncased"
test_size = 0.2
num_labels = 2

In [3]:
import pandas as pd 

df = pd.read_csv(data_set)

df.head()

,email,label
0,date wed NUMBER aug NUMBER NUMBER NUMBER NUMB...,0
1,martin a posted tassos papadopoulos the greek ...,0
2,man threatens explosion in moscow thursday aug...,0
3,klez the virus that won t die already the most...,0
4,in adding cream to spaghetti carbonara which ...,0


In [4]:
from bs4 import BeautifulSoup

def put_line_breaks(text):
    text = str(text).replace('</p>', '</p>\n')
    return text

def remove_html_tags(text):
    cleantext = BeautifulSoup(text, "lxml").text

def clean(text):
    text = put_line_breaks(text)
    text = remove_html_tags(text)

    return text

df['text_cleaned'] = df[text_col].apply(clean)

df.head()



,email,label,text_cleaned
0,date wed NUMBER aug NUMBER NUMBER NUMBER NUMB...,0,None
1,martin a posted tassos papadopoulos the greek ...,0,None
2,man threatens explosion in moscow thursday aug...,0,None
3,klez the virus that won t die already the most...,0,None
4,in adding cream to spaghetti carbonara which ...,0,None


In [5]:
from sklearn import preprocessing

le = preprocessing.LabelEncoder()
le.fit(df[label_col].tolist())
df['label'] = le.transform(df[label_col].tolist())

df.head()

,email,label,text_cleaned
0,date wed NUMBER aug NUMBER NUMBER NUMBER NUMB...,0,None
1,martin a posted tassos papadopoulos the greek ...,0,None
2,man threatens explosion in moscow thursday aug...,0,None
3,klez the virus that won t die already the most...,0,None
4,in adding cream to spaghetti carbonara which ...,0,None


In [6]:
from sklearn.model_selection import train_test_split

df_train, df_test = train_test_split(df,test_size=test_size)

In [7]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(df_train)
test_dataset = Dataset.from_pandas(df_test)

In [8]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name)
def preprocess(examples):
    text_data = [str(text) for text in examples['text_cleaned']]  # Example conversion for safety
    return tokenizer(text_data, truncation=True)

tokenized_train = train_dataset.map(preprocess, batched=True)
tokenized_test = test_dataset.map(preprocess, batched=True)

Map: 100%|██████████| 600/600 [00:00<00:00, 136934.51 examples/s]


In [9]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

model.safetensors: 100%|██████████| 268M/268M [00:02<00:00, 114MB/s]  
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
from transformers import DataCollatorWithPadding
from transformers import TrainingArguments, Trainer
import evaluate
import numpy as np

2024-02-19 18:49:31.410667: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-02-19 18:49:31.430781: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-02-19 18:49:31.430806: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-02-19 18:49:31.431359: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-02-19 18:49:31.435385: I tensorflow/core/platform/cpu_feature_guar

In [11]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [12]:
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis = -1)
    return metric.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    logging_strategy="epoch"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

ImportError: Using the `Trainer` with `PyTorch` requires `accelerate>=0.21.0`: Please run `pip install transformers[torch]` or `pip install accelerate -U`

In [ ]:
trainer.train()
